#### day 4
### Detect data-type issues (numeric-as-text, malformed dates in survey_year, company_founding_year)

In [6]:
import pandas as pd
import numpy as np
company = pd.read_csv("../data/raw/ai_company_adoption.csv")
industry = pd.read_csv("../data/raw/ai_industry_summary.csv")
country = pd.read_csv("../data/raw/country_ai_index.csv")

In [7]:
company.dtypes

response_id                      int64
company_id                      object
survey_year                      int64
quarter                         object
country                         object
region                          object
industry                        object
company_size                    object
num_employees                    int64
annual_revenue_usd_millions    float64
company_founding_year            int64
company_age                      int64
company_age_group               object
ai_adoption_rate               float64
ai_adoption_stage               object
years_using_ai                   int64
ai_primary_tool                 object
num_ai_tools_used                int64
ai_use_case                     object
ai_projects_active               int64
ai_training_hours              float64
ai_budget_percentage           float64
ai_maturity_score              float64
ai_failure_rate                float64
ai_investment_per_employee     float64
regulatory_compliance_sco

#### Part A — Check Numeric-as-Text

In [8]:
numeric_columns = [
    "survey_year",
    "company_founding_year",
    "company_age",
    "ai_adoption_rate",
    "ai_maturity_score"
]

In [9]:
company[numeric_columns].dtypes

survey_year                int64
company_founding_year      int64
company_age                int64
ai_adoption_rate         float64
ai_maturity_score        float64
dtype: object

In [10]:
print(company["survey_year"].min())
print(company["survey_year"].max())

2023
2026


In [11]:
numeric_columns = [
    "survey_year",
    "company_founding_year",
    "company_age",
    "ai_adoption_rate",
    "ai_maturity_score",
    "revenue_growth_percent",
    "cost_reduction_percent"
]

company[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
survey_year,150000.0,2024.500987,1.118013,2023.0,2024.00,2025.000,2026.000,2026.000
company_founding_year,150000.0,2005.993767,9.473220,1990.0,1998.00,2006.000,2014.000,2022.000
company_age,150000.0,18.507220,9.539367,1.0,10.00,18.000,27.000,36.000
ai_adoption_rate,150000.0,36.414726,14.544096,0.0,26.43,36.320,46.200,100.000
ai_maturity_score,150000.0,0.344675,0.133820,0.0,0.25,0.341,0.435,0.904
revenue_growth_percent,150000.0,4.606486,5.318773,-5.0,0.78,4.480,8.240,30.000
cost_reduction_percent,150000.0,4.814909,3.383086,0.0,2.12,4.620,7.130,20.790


In [12]:
print("Survey Year")
print("Minimum:", company["survey_year"].min())
print("Maximum:", company["survey_year"].max())

print("\nCompany Founding Year")
print("Minimum:", company["company_founding_year"].min())
print("Maximum:", company["company_founding_year"].max())

Survey Year
Minimum: 2023
Maximum: 2026

Company Founding Year
Minimum: 1990
Maximum: 2022


#### task 2
#### Detect invalid values / out-of-range in avg_ai_adoption_rate, avg_productivity_change_percent, avg_ai_maturity_score

In [ ]:
#### AI adoption is a percentage.

#### 0 → Company has no AI adoption.
#### 100 → Company has fully adopted AI.

In [13]:
company["ai_adoption_rate"].describe()

count    150000.000000
mean         36.414726
std          14.544096
min           0.000000
25%          26.430000
50%          36.320000
75%          46.200000
max         100.000000
Name: ai_adoption_rate, dtype: float64

### Find Invalid Values

In [14]:
company[(company["ai_adoption_rate"] < 0) |
        (company["ai_adoption_rate"] > 100)]

,response_id,company_id,survey_year,quarter,country,region,industry,company_size,num_employees,annual_revenue_usd_millions,...,productivity_change_percent,jobs_displaced,jobs_created,reskilled_employees,revenue_growth_percent,cost_reduction_percent,innovation_score,customer_satisfaction,survey_source,data_collection_method


--- no invalid values exists

### avg_productivity_change_percent
### -100 = complete productivity loss
### 0 = no change
### 100 = 100% productivity increase

In [17]:
company["productivity_change_percent"].describe()

count    150000.000000
mean          9.266996
std           5.637067
min           0.000000
25%           5.070000
50%           9.060000
75%          13.120000
max          34.360000
Name: productivity_change_percent, dtype: float64

In [18]:
company[(company["productivity_change_percent"] < -100) |
        (company["productivity_change_percent"] > 100)]

,response_id,company_id,survey_year,quarter,country,region,industry,company_size,num_employees,annual_revenue_usd_millions,...,productivity_change_percent,jobs_displaced,jobs_created,reskilled_employees,revenue_growth_percent,cost_reduction_percent,innovation_score,customer_satisfaction,survey_source,data_collection_method


### ai_maturity_score
### This score measures AI maturity.

### 0 = Beginner
### 100 = Highly Mature

### Anything outside this range is invalid.

In [19]:
company["ai_maturity_score"].describe()

count    150000.000000
mean          0.344675
std           0.133820
min           0.000000
25%           0.250000
50%           0.341000
75%           0.435000
max           0.904000
Name: ai_maturity_score, dtype: float64

In [20]:
company[(company["ai_maturity_score"] < 0) |
        (company["ai_maturity_score"] > 100)]

,response_id,company_id,survey_year,quarter,country,region,industry,company_size,num_employees,annual_revenue_usd_millions,...,productivity_change_percent,jobs_displaced,jobs_created,reskilled_employees,revenue_growth_percent,cost_reduction_percent,innovation_score,customer_satisfaction,survey_source,data_collection_method


### task 3 : Detect inconsistent categories in region, survey_source


In [21]:
company["region"].unique()

array(['Europe', 'North America', 'South America', 'Asia', 'Africa',
       'Oceania'], dtype=object)

In [22]:
company["survey_source"].unique()

array(['WEF Survey', 'McKinsey Report', 'Internal Corporate Survey',
       'LinkedIn Poll'], dtype=object)

In [23]:
company["region"].value_counts()

region
Asia             49727
Europe           40835
South America    19909
Africa           19787
Oceania          10002
North America     9740
Name: count, dtype: int64

In [24]:
company["survey_source"].value_counts()

survey_source
LinkedIn Poll                37727
Internal Corporate Survey    37597
McKinsey Report              37479
WEF Survey                   37197
Name: count, dtype: int64

In [25]:
### Remove extra spaces and standardize the text.

In [26]:
company["region"] = company["region"].str.strip()
company["region"].value_counts()

region
Asia             49727
Europe           40835
South America    19909
Africa           19787
Oceania          10002
North America     9740
Name: count, dtype: int64

In [27]:
company["survey_source"] = company["survey_source"].str.strip()
company["survey_source"].value_counts()

survey_source
LinkedIn Poll                37727
Internal Corporate Survey    37597
McKinsey Report              37479
WEF Survey                   37197
Name: count, dtype: int64

In [29]:
### Check Case Consistency
### Sometimes datasets contain:

In [30]:
company["region"].str.lower().value_counts()

region
asia             49727
europe           40835
south america    19909
africa           19787
oceania          10002
north america     9740
Name: count, dtype: int64

In [31]:
company["survey_source"].str.lower().value_counts()

survey_source
linkedin poll                37727
internal corporate survey    37597
mckinsey report              37479
wef survey                   37197
Name: count, dtype: int64